# XGBoost — The Algorithm That Wins Competitions

## What is XGBoost?

XGBoost (eXtreme Gradient Boosting) is a **highly optimized gradient boosting library** that became famous for winning a huge proportion of Kaggle competitions from 2014–2018. It implements gradient boosted decision trees with a focus on speed, performance, and scalability.

**Real-world analogy**: Imagine you're trying to predict tomorrow's weather. One meteorologist makes predictions and writes down where they were wrong. A second meteorologist studies those mistakes and makes better predictions. A third studies the second's mistakes. XGBoost is this chain — each tree corrects the previous trees' errors, building up to a highly accurate ensemble.

## XGBoost vs sklearn's GradientBoosting

| Feature | sklearn GBM | XGBoost |
|---------|-------------|--------|
| Speed | Slow | 5-10× faster |
| Regularization | Basic | L1 + L2 + tree |
| Missing values | Error | Handles natively |
| Parallel training | No | Yes |
| GPU support | No | Yes |
| Early stopping | Manual | Built-in |

## Prerequisites
- Scikit-Learn basics (train_test_split, metrics)
- Understanding of decision trees and gradient boosting

## Table of Contents
1. Installation & Two APIs
2. How Gradient Boosting Works (Intuition)
3. XGBoost Sklearn API (XGBClassifier / XGBRegressor)
4. XGBoost Native API (DMatrix + xgb.train)
5. Key Hyperparameters Explained
6. Handling Missing Values
7. Early Stopping
8. Feature Importance
9. Regularization to Prevent Overfitting
10. Cross-Validation with xgb.cv
11. Common Pitfalls
12. Mini Project: Titanic Survival Prediction
13. Interview Q&A
14. Resources

---

**Official Docs**: https://xgboost.readthedocs.io/  
**XGBoost Paper**: Chen & Guestrin (2016) — https://arxiv.org/abs/1603.02754  
**YouTube (StatQuest)**: https://www.youtube.com/watch?v=OtD8wVaFm6E  
**YouTube (Sentdex XGBoost)**: https://www.youtube.com/watch?v=OQKQHNCVf5k

In [ ]:
xgboosttry:
    import xgboost as xgb
    from xgboost import XGBClassifier, XGBRegressor
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
    from sklearn.datasets import load_breast_cancer, make_classification
    from sklearn.metrics import roc_auc_score
    import warnings; warnings.filterwarnings("ignore")
    print(f"XGBoost {xgb.__version__} ready")
except ImportError:
    raise SystemExit("Run: pip install xgboost scikit-learn pandas numpy matplotlib")


---
## 2. How Gradient Boosting Works (Intuition)

### Step-by-Step Intuition

1. **Start with a simple prediction** — e.g., predict the average house price for everyone
2. **Calculate residuals** — for each house, how wrong was your prediction?
3. **Train a decision tree to predict the residuals** — the tree tries to explain the errors
4. **Update predictions** = old prediction + learning_rate × tree prediction
5. **Repeat steps 2-4** for N rounds (n_estimators)

**What makes XGBoost special over basic gradient boosting?**
- Uses **regularized objective function**: loss + L1/L2 penalties on leaf weights
- **Approximate split finding**: can find splits in parallel (not sequentially)
- **Cache-aware access**: optimizes memory usage for speed
- **Sparsity-aware**: handles missing values and sparse data efficiently

### Two APIs
1. **Sklearn API** (`XGBClassifier`, `XGBRegressor`) — drop-in replacement for sklearn models
2. **Native API** (`xgb.DMatrix`, `xgb.train`) — more control, faster for large data

In [ ]:
# ── Sklearn API: exact same pattern as sklearn models ──────────────────────────
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                      random_state=42, stratify=y)

# XGBClassifier works exactly like sklearn's classifiers!
xgb_clf = XGBClassifier(
    n_estimators=200,       # Number of trees (boosting rounds)
    learning_rate=0.1,      # Shrinkage factor (smaller = more conservative)
    max_depth=4,            # Max tree depth (3-6 is typical)
    min_child_weight=1,     # Minimum sum of instance weight in a leaf
    subsample=0.8,          # Fraction of samples used per tree
    colsample_bytree=0.8,   # Fraction of features used per tree
    gamma=0,                # Min loss reduction to split a node
    reg_alpha=0,            # L1 regularization
    reg_lambda=1,           # L2 regularization
    random_state=42,
    eval_metric='logloss',  # Metric for evaluation during training
    verbosity=0             # Suppress output
)

xgb_clf.fit(X_train, y_train)
y_pred  = xgb_clf.predict(X_test)
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

print("=== XGBClassifier (Sklearn API) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}")

# XGBRegressor for regression
X_reg, y_reg = make_regression(n_samples=1000, n_features=15, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

xgb_reg = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
xgb_reg.fit(X_tr, y_tr)
preds_reg = xgb_reg.predict(X_te)
print(f"\n=== XGBRegressor ===")
print(f"RMSE: {np.sqrt(mean_squared_error(y_te, preds_reg)):.4f}")
print(f"R²:   {r2_score(y_te, preds_reg):.4f}")

In [ ]:
# ── Native API: DMatrix + xgb.train ────────────────────────────────────────────
# DMatrix is XGBoost's optimized data structure (faster than numpy arrays for large data)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(cancer.feature_names))
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=list(cancer.feature_names))

params = {
    'objective':  'binary:logistic',  # Output probabilities for binary classification
    'eval_metric':'auc',
    'max_depth':  4,
    'eta':        0.1,           # Same as learning_rate
    'subsample':  0.8,
    'colsample_bytree': 0.8,
    'seed':       42,
}

evals_result = {}  # Will capture training history
model_native = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, 'train'), (dtest, 'eval')],
    evals_result=evals_result,
    early_stopping_rounds=20,  # Stop if no improvement for 20 rounds
    verbose_eval=False
)

print(f"=== Native API ===")
print(f"Best iteration: {model_native.best_iteration}")
print(f"Best eval AUC:  {model_native.best_score:.4f}")

# Plot training curves
plt.figure(figsize=(9, 4))
plt.plot(evals_result['train']['auc'], label='Train AUC', color='steelblue')
plt.plot(evals_result['eval']['auc'], label='Val AUC', color='red')
plt.axvline(model_native.best_iteration, color='green', linestyle='--',
             label=f'Best iteration ({model_native.best_iteration})')
plt.title('XGBoost Training Curves (AUC)')
plt.xlabel('Boosting Round')
plt.ylabel('AUC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print("If train>>val: overfitting. Use early stopping or more regularization.")

---
## 5. Key Hyperparameters Explained

| Parameter | Effect | Typical Range | To reduce overfitting |
|-----------|--------|---------------|-----------------------|
| `n_estimators` | More trees → more accurate but slower | 100-2000 | Use early stopping |
| `learning_rate` (eta) | Lower → more conservative, needs more trees | 0.01-0.3 | Lower value |
| `max_depth` | Deeper trees → more complex, more overfit | 3-8 | Reduce |
| `min_child_weight` | Higher → simpler trees | 1-10 | Increase |
| `subsample` | Fraction of rows per tree | 0.5-1.0 | Reduce |
| `colsample_bytree` | Fraction of columns per tree | 0.5-1.0 | Reduce |
| `gamma` | Min gain to create a split | 0-5 | Increase |
| `reg_alpha` | L1 regularization | 0-1 | Increase |
| `reg_lambda` | L2 regularization | 1-10 | Increase |

**Golden Rule**: Start with default params, then:
1. Tune `max_depth` and `min_child_weight` (biggest impact)
2. Tune `subsample` and `colsample_bytree` (reduce overfitting)
3. Tune `reg_alpha` / `reg_lambda` (further regularize)
4. Lower `learning_rate` and increase `n_estimators` (squeeze last accuracy)

In [ ]:
# ── XGBoost handles missing values natively! ───────────────────────────────────
# Introduce 20% missing values
np.random.seed(42)
X_miss = X_train.copy().astype(float)
mask   = np.random.random(X_miss.shape) < 0.20  # 20% missing
X_miss[mask] = np.nan

X_te_miss = X_test.copy().astype(float)
mask_te   = np.random.random(X_te_miss.shape) < 0.20
X_te_miss[mask_te] = np.nan

# Train XGBoost with missing data — NO imputation needed!
xgb_miss = XGBClassifier(n_estimators=200, random_state=42, verbosity=0)
xgb_miss.fit(X_miss, y_train)
pred_miss = xgb_miss.predict(X_te_miss)

print("=== XGBoost: Native Missing Value Handling ===")
print(f"Data missing rate: 20%")
print(f"No imputation needed — XGBoost handles it internally!")
print(f"Accuracy with 20% missing: {accuracy_score(y_test, pred_miss):.4f}")
print(f"Accuracy without missing:  {accuracy_score(y_test, xgb_clf.predict(X_test)):.4f}")
print(f"\nHow: XGBoost learns the optimal default direction for missing values during training.")

# ── Feature Importance ─────────────────────────────────────────────────────────
importance_types = ['weight', 'gain', 'cover']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, imp_type in zip(axes, importance_types):
    importance = xgb_clf.get_booster().get_score(importance_type=imp_type)
    # Sort and take top 10
    top10 = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
    names, vals = zip(*top10)
    ax.barh(list(names)[::-1], list(vals)[::-1], color='steelblue', alpha=0.8)
    ax.set_title(f'Importance type: {imp_type}')
    ax.set_xlabel(imp_type.capitalize())

plt.suptitle('XGBoost Feature Importance: 3 Types', fontweight='bold')
plt.tight_layout()
plt.show()

print("Importance types:")
print("  weight: number of times feature is used in a split")
print("  gain:   average training loss reduction per split (most informative)")
print("  cover:  avg number of samples affected by feature splits")

In [ ]:
# ── xgb.cv: Cross-Validation with Native API ───────────────────────────────────
# This is particularly good for finding the optimal n_estimators!
from sklearn.datasets import load_breast_cancer

X_full, y_full = load_breast_cancer(return_X_y=True)
dtrain_full = xgb.DMatrix(X_full, label=y_full)

params = {
    'objective':  'binary:logistic',
    'eval_metric':'auc',
    'max_depth':  4,
    'eta':        0.05,
    'subsample':  0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

cv_results = xgb.cv(
    params,
    dtrain_full,
    num_boost_round=500,
    nfold=5,
    early_stopping_rounds=30,
    verbose_eval=False
)

best_round = cv_results['test-auc-mean'].idxmax()
best_auc   = cv_results['test-auc-mean'].max()
print(f"=== xgb.cv (5-fold) ===")
print(f"Best n_estimators: {best_round + 1}")
print(f"Best CV AUC: {best_auc:.4f} ± {cv_results['test-auc-std'][best_round]:.4f}")

# Plot CV results
plt.figure(figsize=(9, 4))
plt.plot(cv_results.index, cv_results['train-auc-mean'], 'b-', label='Train AUC')
plt.fill_between(cv_results.index,
                  cv_results['train-auc-mean'] - cv_results['train-auc-std'],
                  cv_results['train-auc-mean'] + cv_results['train-auc-std'], alpha=0.2)
plt.plot(cv_results.index, cv_results['test-auc-mean'], 'r-', label='Val AUC (5-fold CV)')
plt.fill_between(cv_results.index,
                  cv_results['test-auc-mean'] - cv_results['test-auc-std'],
                  cv_results['test-auc-mean'] + cv_results['test-auc-std'], alpha=0.2, color='red')
plt.axvline(best_round, color='green', linestyle='--', label=f'Best: round {best_round+1}')
plt.title('XGBoost CV: Finding Optimal n_estimators')
plt.xlabel('Boosting Round')
plt.ylabel('AUC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 11. Common Pitfalls

| Pitfall | Problem | Fix |
|---------|---------|-----|
| No early stopping | Overfits as n_estimators grows | Use `early_stopping_rounds=20` with eval set |
| `learning_rate` too high | Unstable training, poor generalization | Start at 0.1, reduce to 0.01-0.05 for best results |
| Imbalanced classes | Model predicts majority class | `scale_pos_weight = n_neg/n_pos` |
| Not tuning `max_depth` | Default depth=6 often overfits | Start with depth=3-5, tune from there |
| Using `weight` importance | Number of splits, not actually important | Use `gain` importance instead |
| Forgetting `verbosity=0` | Lots of output clutters notebook | Set `verbosity=0` for sklearn API |

---
## 12. Mini Project: Titanic Survival Prediction

The classic Kaggle Titanic problem: predict who survived the Titanic disaster. This requires feature engineering, handling missing values, and XGBoost tuning.

In [ ]:
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt

np.random.seed(42)

# ── Simulate Titanic-like data ─────────────────────────────────────────────────
n = 891
df = pd.DataFrame({
    'Pclass':   np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
    'Sex':      np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
    'Age':      np.random.normal(29.7, 14.5, n).clip(1, 80),
    'SibSp':    np.random.choice(range(9), n, p=[0.68, 0.23, 0.04, 0.02, 0.01, 0.01, 0.005, 0.0025, 0.0025]),
    'Parch':    np.random.choice(range(7), n, p=[0.76, 0.13, 0.07, 0.02, 0.01, 0.005, 0.005]),
    'Fare':     np.random.exponential(32, n).clip(0, 512),
    'Embarked': np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09]),
})

# Add missing values
df.loc[df.sample(177, random_state=1).index, 'Age'] = np.nan  # ~20% missing
df.loc[df.sample(2, random_state=2).index, 'Embarked'] = np.nan

# Realistic survival probability (women+children+1st class more likely to survive)
score = (
    (df['Sex'] == 'female').astype(float) * 2.5
    - (df['Pclass'] - 1) * 0.8
    + (df['Age'].fillna(29.7) < 16).astype(float) * 0.8
    - df['Fare'].fillna(32) / 200
    + np.random.randn(n) * 0.8
)
y = (score > score.median()).astype(int)
print(f"Survival rate: {y.mean():.1%}")

# ── Feature Engineering ────────────────────────────────────────────────────────
def engineer_features(df):
    df = df.copy()
    df['Sex']          = (df['Sex'] == 'female').astype(int)  # Encode gender
    df['Age']          = df['Age'].fillna(df['Age'].median())  # Impute age
    df['Embarked']     = df['Embarked'].fillna('S').map({'S': 0, 'C': 1, 'Q': 2})
    df['FamilySize']   = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']      = (df['FamilySize'] == 1).astype(int)
    df['FarePerPerson']= df['Fare'] / df['FamilySize']
    df['AgeClass']     = df['Age'] * df['Pclass']  # Interaction feature
    df['IsChild']      = (df['Age'] < 16).astype(int)
    return df

df_eng = engineer_features(df)
feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
                'FamilySize', 'IsAlone', 'FarePerPerson', 'AgeClass', 'IsChild']
X_all = df_eng[feature_cols]

# ── Train/Test Split ──────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y, test_size=0.2, 
                                            random_state=42, stratify=y)

# ── Train XGBoost with Early Stopping ─────────────────────────────────────────
model = XGBClassifier(
    n_estimators=1000,          # High — early stopping will find optimal
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.5,
    scale_pos_weight=1,         # For imbalanced: n_negative/n_positive
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=30,
    verbosity=0
)

model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
print(f"Best iteration: {model.best_iteration}")

y_pred_test  = model.predict(X_te)
y_proba_test = model.predict_proba(X_te)[:, 1]

print(f"\n=== Final Test Performance ===")
print(f"Accuracy:  {accuracy_score(y_te, y_pred_test):.4f}")
print(f"F1 Score:  {f1_score(y_te, y_pred_test):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_te, y_proba_test):.4f}")
print(f"\n{classification_report(y_te, y_pred_test, target_names=['Did not survive', 'Survived'])}")

# ── Feature Importance ─────────────────────────────────────────────────────────
imp = pd.Series(model.get_booster().get_score(importance_type='gain'),
                name='Gain Importance').sort_values(ascending=True)

plt.figure(figsize=(8, 5))
imp.plot(kind='barh', color='steelblue', alpha=0.8)
plt.title('Feature Importance (Gain): Titanic Survival', fontweight='bold')
plt.xlabel('Gain')
plt.tight_layout()
plt.show()

top_feat = imp.sort_values(ascending=False).index[0]
print(f"\nMost important feature: {top_feat}")
print(f"Engineered features (FamilySize, AgeClass, etc.) boosted the model!")

---
## 13. Interview Q&A

**Q1: What is the key difference between XGBoost and Random Forest?**  
**A**: Random Forest uses **bagging** — builds trees in parallel on random subsets, then averages. XGBoost uses **boosting** — builds trees sequentially, each one correcting the errors of the previous ones. Boosting typically achieves higher accuracy but is more prone to overfitting if not regularized properly. Random Forest is more robust and parallelizes more easily.

---
**Q2: What is the `learning_rate` in XGBoost and what is its effect?**  
**A**: `learning_rate` (eta) is a shrinkage factor (0-1) applied to each tree's predictions before adding them to the ensemble. Lower learning rate = each tree contributes less = need more trees to converge, but the model generalizes better. Rule of thumb: use `learning_rate=0.05-0.1` and tune `n_estimators` with early stopping.

---
**Q3: What is the `scale_pos_weight` parameter for?**  
**A**: For imbalanced binary classification. Set to `n_negative/n_positive` to give more weight to the minority class. For example, if you have 90% negative and 10% positive, set `scale_pos_weight=9`.

---
**Q4: What is early stopping and why is it important?**  
**A**: Early stopping monitors model performance on a validation set during training and stops adding more trees once performance stops improving. It prevents overfitting and automatically finds the optimal `n_estimators`. Set `early_stopping_rounds=20` — training stops if no improvement after 20 consecutive rounds.

---
**Q5: How does XGBoost handle missing values?**  
**A**: XGBoost learns the optimal default direction (left or right branch) for missing values at each split. During training, it tries both directions and picks the one that reduces loss most. This means you can pass data with NaN values directly to XGBoost without imputation — it's a built-in feature, not a workaround.

---
## 14. Resources

- **Official Docs**: https://xgboost.readthedocs.io/
- **XGBoost Paper** (Chen & Guestrin 2016): https://arxiv.org/abs/1603.02754
- **StatQuest — XGBoost**: https://www.youtube.com/watch?v=OtD8wVaFm6E (best intuition)
- **Kaggle Tutorials**: https://www.kaggle.com/learn (many use XGBoost)
- **XGBoost Python Tutorial**: https://xgboost.readthedocs.io/en/stable/python/python_intro.html

---
## Summary & What's Next

| Concept | Key Point |
|---------|----------|
| Gradient Boosting | Sequential tree building; each tree corrects previous errors |
| Sklearn API | `XGBClassifier` / `XGBRegressor` — drop-in sklearn replacement |
| Native API | `DMatrix` + `xgb.train` — more control, faster for large data |
| Missing values | XGBoost handles NaN natively — no imputation needed |
| Early stopping | `early_stopping_rounds=20` — automatically finds optimal n_estimators |
| Feature importance | Use `gain` type (not `weight`) for meaningful importance |
| Regularization | `reg_alpha` (L1), `reg_lambda` (L2), `gamma`, `min_child_weight` |

**Next**: LightGBM — XGBoost's faster sibling, especially for large datasets.